# Data Quality — Dengue SINAN (CasoDengue)

In [ ]:
import os
import matplotlib.pyplot as plt
import pandas
from sqlalchemy import create_engine, text
from pandas import DataFrame
from IPython.display import Markdown, display

# 2. Data Loading

In [ ]:
engine = create_engine(os.environ["DATABASE_URL"])

with engine.connect() as conn:
    dataset: DataFrame = pandas.read_sql(text("SELECT * FROM caso_dengue"), conn)

display(Markdown(f"**Total rows:** {len(dataset):,}"))
dataset.head(5)

## 2.1 Schema Overview

In [ ]:
schema = DataFrame(
    {
        "column": dataset.columns,
        "dtype": dataset.dtypes.values,
        "non_null": dataset.notna().sum().values,
        "null_count": dataset.isna().sum().values,
        "null_pct": (dataset.isna().mean() * 100).round(2).values,
    }
)
schema

## 2.2 Memory Usage

In [ ]:
total_mb = dataset.memory_usage(deep=True).sum() / 1024**2
display(Markdown(f"**Total memory usage:** {total_mb:.2f} MB"))

col_memory = (
    dataset.memory_usage(deep=True)
    .drop("Index")
    .sort_values(ascending=False)
    .reset_index()
)
col_memory.columns = ["column", "bytes"]
col_memory["MB"] = (col_memory["bytes"] / 1024**2).round(3)
col_memory[["column", "MB"]]

# 3. Missing Values Analysis

In [ ]:
missing = dataset.isna().sum().sort_values(ascending=False)
missing_pct = (dataset.isna().mean() * 100).sort_values(ascending=False)

missing_df = DataFrame({"null_count": missing, "null_pct_%": missing_pct.round(2)})
missing_df = missing_df[missing_df["null_count"] > 0]

display(
    Markdown(
        f"**Columns with missing values:** {len(missing_df)} / {len(dataset.columns)}"
    )
)
display(Markdown(f"**Total missing cells:** {missing.sum():,}"))
missing_df

## 3.1 Missing Values by Column (bar chart)

In [ ]:
if not missing_df.empty:
    fig, ax = plt.subplots(figsize=(10, max(3, len(missing_df) * 0.5)))
    ax.barh(missing_df.index, missing_df["null_pct_%"], color="#ef9a9a", alpha=0.8)
    ax.set_xlabel("Missing (%)")
    ax.set_title("Missing Values per Column", fontsize=13, pad=12)
    ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()
    plt.show()
else:
    display(Markdown("**No missing values found.**"))

# 4. Data Integrity

Checks that coded columns contain only valid SINAN values.

In [ ]:
VALID_OUTCOME = {1, 2, 3, 4, 9}

outcome_counts = dataset["outcome"].value_counts(dropna=False).sort_index()
invalid_outcome = dataset[
    ~dataset["outcome"].isin(VALID_OUTCOME) & dataset["outcome"].notna()
]

display(Markdown("**Outcome value distribution:**"))
display(outcome_counts.rename("count").to_frame())
display(Markdown(f"**Rows with invalid outcome code:** {len(invalid_outcome):,}"))

## 4.1 Outcome codes

Valid SINAN values: `1` = Cure, `2` = Death by dengue, `3` = Death by other cause, `4` = Ongoing, `9` = Unknown/ignored.

In [ ]:
VALID_SEROTYPE = {1, 2, 3, 4, 5}

serotype_counts = dataset["serotype"].value_counts(dropna=False).sort_index()
invalid_serotype = dataset[
    ~dataset["serotype"].isin(VALID_SEROTYPE) & dataset["serotype"].notna()
]

display(Markdown("**Serotype value distribution:**"))
display(serotype_counts.rename("count").to_frame())
display(Markdown(f"**Rows with invalid serotype code:** {len(invalid_serotype):,}"))

## 4.2 Serotype codes

Valid SINAN values: `1` = DENV-1, `2` = DENV-2, `3` = DENV-3, `4` = DENV-4, `5` = Unclassified/unknown.

## 4.3 Hospitalized codes

Valid SINAN values: `1` = Yes, `2` = No, `9` = Unknown/ignored.

In [ ]:
VALID_HOSPITALIZED = {1, 2, 9}

hosp_counts = dataset["hospitalized"].value_counts(dropna=False).sort_index()
invalid_hosp = dataset[
    ~dataset["hospitalized"].isin(VALID_HOSPITALIZED) & dataset["hospitalized"].notna()
]

display(Markdown("**Hospitalized value distribution:**"))
display(hosp_counts.rename("count").to_frame())
display(Markdown(f"**Rows with invalid hospitalized code:** {len(invalid_hosp):,}"))

## 4.4 Notification date range

In [ ]:
dataset["notification_date"] = pandas.to_datetime(dataset["notification_date"])

display(
    Markdown(f"**Earliest notification:** {dataset['notification_date'].min().date()}")
)
display(
    Markdown(f"**Latest notification:** {dataset['notification_date'].max().date()}")
)

future_dates = dataset[dataset["notification_date"] > pandas.Timestamp.today()]
display(Markdown(f"**Rows with future dates:** {len(future_dates):,}"))

# 5. Data Coverage by State

In [ ]:
IBGE_TO_STATE: dict[int, str] = {
    11: "RO",
    12: "AC",
    13: "AM",
    14: "RR",
    15: "PA",
    16: "AP",
    17: "TO",
    21: "MA",
    22: "PI",
    23: "CE",
    24: "RN",
    25: "PB",
    26: "PE",
    27: "AL",
    28: "SE",
    29: "BA",
    31: "MG",
    32: "ES",
    33: "RJ",
    35: "SP",
    41: "PR",
    42: "SC",
    43: "RS",
    50: "MS",
    51: "MT",
    52: "GO",
    53: "DF",
}

coverage = (
    dataset.groupby("state_ibge_code")["notification_date"]
    .agg(start="min", end="max", total_notifications="count")
    .reset_index()
)
coverage["state"] = coverage["state_ibge_code"].map(IBGE_TO_STATE)
coverage["days_span"] = (coverage["end"] - coverage["start"]).dt.days + 1

display(Markdown(f"**States present:** {len(coverage)} / 27"))
coverage[["state", "start", "end", "days_span", "total_notifications"]].sort_values(
    "total_notifications", ascending=False
)

# 6. Decision Summary

| Column / Issue | Problem | Decision |
|---|---|---|
| `outcome` nulls | Not all notifications have a recorded outcome | Accept — outcome is optional in SINAN; null ≠ no event |
| `serotype` nulls | Serotyping is not performed for every case | Accept — serotype is optional; null = untyped |
| `hospitalized` nulls | Not all records include hospitalization status | Accept — treat null as unknown, not as "not hospitalized" |
| `age_encoded` nulls | Age not recorded for some notifications | Accept — exclude nulls when computing age statistics |
| `death_date` nulls | Only populated when outcome = 2 (death) | Expected — not a data quality issue |
| `state_ibge_code` / `municipality_ibge_code` | Must be non-null; IBGE codes drive all aggregations | Flag any null; these rows should be investigated before use |

In [ ]:
critical_cols = ["state_ibge_code", "municipality_ibge_code", "notification_date"]
critical_missing = dataset[critical_cols].isna().sum()

display(Markdown("**Missing values in critical columns:**"))
display(critical_missing.rename("null_count").to_frame())